# NB3 — train correct LoRA, measured NB2 included
Upload this notebook to Colab, select T4 GPU, then Run all. No GitHub push required.
Uses frozen baseline (b) target=0.84, full eval and the proved NB1 mask. Does not rerun NB2.
Download ZIP includes adapters/correct/, results/ and data/split/. Extract into the repo root.


In [ ]:
import base64, hashlib, io, os, pathlib, subprocess, sys, zipfile
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == 'd01651462eaf0ab29da4dd10ed2c06afd4388730fa6d05726be9f8498c743b8d'
root = pathlib.Path("/content/lab21_nb3_portable")
root.mkdir(exist_ok=True)
assert not (root / "adapters/correct/adapter_config.json").exists(), "Adapter already saved here; run the download cell"
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(root)
os.chdir(root)
os.environ.update(COMPUTE_TIER="T4", MASK_MODE="assistant-only", EPOCHS="2")
os.environ.pop("BASE_MODEL", None)
os.environ.pop("EVAL_LIMIT", None)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Source + frozen NB2 bundle:", 'd01651462eaf0ab29da4dd10ed2c06afd4388730fa6d05726be9f8498c743b8d')


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-u", "scripts/colab_run.py", "nb3"], check=True)


In [ ]:
import csv, json, pathlib, zipfile
from google.colab import files
root = pathlib.Path.cwd()
adapter = root / "adapters/correct"
assert (adapter / "adapter_config.json").exists(), "Training has not saved the adapter yet"
assert (adapter / "adapter_model.safetensors").exists(), "Adapter weights missing"
with (root / "results/runs.csv").open() as fh:
    rows = list(csv.DictReader(fh))
print(json.dumps(rows, indent=2))
out = root / "nb3_results_and_adapter.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as archive:
    for folder in (root / "results", adapter, root / "data/split"):
        for path in folder.rglob("*"):
            if path.is_file():
                archive.write(path, path.relative_to(root))
files.download(str(out))
print("Saved adapter and measured training results. Keep baseline and eval unchanged for NB4/NB5.")
